# 🏥 Uncertainty-Aware Medical VQA — Complete Pipeline

**All-in-one notebook:** Baseline → Fine-Tuning → Uncertainty → Abstention → Safety Analysis

### Skip Flags
Each phase can be **skipped** by setting its flag to `True` in Cell 2.
- Skipped phases load saved results from disk instead of re-running.
- This means you can run Phase 1 today, close the notebook, and resume from Phase 2 tomorrow.

| Phase | What It Does | Time (T4) | Skip Flag |
|-------|-------------|-----------|----------|
| **1. Baseline** | Zero-shot BLIP-2 inference | ~10 min | `SKIP_BASELINE` |
| **2. Fine-Tuning** | LoRA training on stratified subset | ~30 min | `SKIP_TRAINING` |
| **3. Uncertainty** | Entropy + MC Dropout + Abstention | ~30 min | `SKIP_UNCERTAINTY` |
| **4. Comparison** | Final tables + safety plots | ~2 min | Always runs |

**Requirements:** T4 GPU (`Runtime > Change runtime type > T4 GPU`)

---
## Cell 1: Install Dependencies

In [ ]:
%pip install -q transformers accelerate peft bitsandbytes datasets Pillow tqdm pandas scikit-learn nltk bert-score matplotlib

## Cell 2: Configuration & Skip Flags

In [ ]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# ============================================================
# SKIP FLAGS — Set True to skip a phase and load saved results
# ============================================================
SKIP_BASELINE    = False   # Skip Phase 1: load baseline_summary.json
SKIP_TRAINING    = False   # Skip Phase 2: load LoRA checkpoint
SKIP_UNCERTAINTY = False   # Skip Phase 3: load uncertainty_summary.json

# ============================================================
# PROJECT PATHS
# ============================================================
USE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector"

PROJECT_DIR = DRIVE_PROJECT_DIR if USE_DRIVE else "/content/kvasir-vqa"
DATA_DIR    = f"{PROJECT_DIR}/data"
IMAGE_DIR   = f"{DATA_DIR}/images"
RESULTS_DIR = f"{PROJECT_DIR}/results"
PRED_DIR    = f"{RESULTS_DIR}/predictions"
UNC_DIR     = f"{RESULTS_DIR}/uncertainty"
CKPT_DIR    = f"{PROJECT_DIR}/checkpoints"

for d in [DATA_DIR, IMAGE_DIR, PRED_DIR, UNC_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# MODEL & TRAINING CONFIG
# ============================================================
MODEL_NAME         = "Salesforce/blip2-opt-2.7b"
SEED               = 42
MAX_NEW_TOKENS     = 64
EVAL_SAMPLES       = 50

# Fine-tuning
TRAIN_SUBSET_SIZE  = 2000
EPOCHS             = 3
BATCH_SIZE         = 4
GRAD_ACCUM_STEPS   = 4
LEARNING_RATE      = 2e-4
WEIGHT_DECAY       = 0.01
WARMUP_RATIO       = 0.1

# LoRA
LORA_R              = 16
LORA_ALPHA           = 32
LORA_DROPOUT         = 0.1
LORA_TARGET_MODULES  = ["q_proj", "v_proj"]

# Uncertainty
MC_DROPOUT_PASSES  = 5
TARGET_COVERAGE    = 0.80

print("Configuration:")
print(f"  Project:    {PROJECT_DIR}")
print(f"  Model:      {MODEL_NAME}")
print(f"  Train size: {TRAIN_SUBSET_SIZE} | Eval size: {EVAL_SAMPLES}")
print(f"  LoRA:       r={LORA_R}, \u03b1={LORA_ALPHA}")
print(f"  MC Passes:  {MC_DROPOUT_PASSES} | Coverage: {TARGET_COVERAGE*100:.0f}%")
print(f"  Skipping:   Baseline={SKIP_BASELINE}, Training={SKIP_TRAINING}, Uncertainty={SKIP_UNCERTAINTY}")

## Cell 3: Download / Load Data

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

train_csv = Path(DATA_DIR) / "kvasir_vqa_x1_train.csv"
test_csv  = Path(DATA_DIR) / "kvasir_vqa_x1_test.csv"

if train_csv.exists() and test_csv.exists():
    print(f"Data found at {DATA_DIR}")
    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)
else:
    print("Downloading dataset...")
    from datasets import load_dataset
    from tqdm.auto import tqdm

    ds = load_dataset("SimulaMet/Kvasir-VQA-x1")
    train_df = ds['train'].to_pandas()
    test_df  = ds['test'].to_pandas()
    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)

    print("Downloading images...")
    img_ds = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
    for sample in tqdm(img_ds, desc="Saving images"):
        img_id = sample.get('img_id', sample.get('imgId', ''))
        img = sample.get('img', sample.get('image', None))
        if img is not None and img_id:
            save_path = Path(IMAGE_DIR) / f"{img_id}.jpg"
            if not save_path.exists():
                img.save(str(save_path))

n_images = len(list(Path(IMAGE_DIR).glob('*.jpg')))
print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Images: {n_images:,}")

# Create eval subset (same for all phases — apples-to-apples comparison)
np.random.seed(SEED)
eval_subset = test_df.sample(n=min(EVAL_SAMPLES, len(test_df)), random_state=SEED)
print(f"Eval subset: {len(eval_subset)} samples")

## Cell 4: All Metric & Utility Functions

Shared across all phases.

In [ ]:
import torch
from PIL import Image
from tqdm.auto import tqdm

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor

# ── Text Normalization ──
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

# ── VQA Metrics ──
def compute_exact_match(pred, gt):
    return normalize_text(pred) == normalize_text(gt)

def compute_word_f1(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not p_tok and not g_tok: return 1.0
    if not p_tok or not g_tok: return 0.0
    common = sum((Counter(p_tok) & Counter(g_tok)).values())
    if common == 0: return 0.0
    prec = common / len(p_tok)
    rec  = common / len(g_tok)
    return 2 * prec * rec / (prec + rec)

def compute_word_precision(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not p_tok: return 1.0 if not g_tok else 0.0
    common = sum((Counter(p_tok) & Counter(g_tok)).values())
    return common / len(p_tok)

def compute_word_recall(pred, gt):
    """Critical in medical domain — missing a finding is worse than false alarm."""
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not g_tok: return 1.0
    if not p_tok: return 0.0
    common = sum((Counter(p_tok) & Counter(g_tok)).values())
    return common / len(g_tok)

def compute_bleu_scores(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not g_tok: return {f'bleu_{n}': (1.0 if not p_tok else 0.0) for n in range(1,5)}
    if not p_tok: return {f'bleu_{n}': 0.0 for n in range(1,5)}
    smooth = SmoothingFunction().method1
    scores = {}
    for n in range(1, 5):
        w = tuple([1.0/n]*n + [0.0]*(4-n))
        scores[f'bleu_{n}'] = sentence_bleu([g_tok], p_tok, weights=w, smoothing_function=smooth)
    return scores

def compute_rouge_l(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not p_tok and not g_tok: return 1.0
    if not p_tok or not g_tok: return 0.0
    m, n = len(g_tok), len(p_tok)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if g_tok[i-1]==p_tok[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    if lcs == 0: return 0.0
    prec = lcs / n
    rec  = lcs / m
    return 2 * prec * rec / (prec + rec)

def compute_meteor(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not g_tok: return 1.0 if not p_tok else 0.0
    if not p_tok: return 0.0
    return nltk_meteor([g_tok], p_tok)

# ── Evaluate a list of predictions ──
def evaluate_predictions(predictions, ground_truths, complexities, label="Model"):
    """Compute all metrics. Returns a results dict."""
    em_l, f1_l, prec_l, rec_l = [], [], [], []
    bl1_l, bl2_l, bl3_l, bl4_l = [], [], [], []
    rl_l, met_l = [], []
    per_comp = {}

    for pred, gt, comp in zip(predictions, ground_truths, complexities):
        em  = int(compute_exact_match(pred, gt))
        f1  = compute_word_f1(pred, gt)
        p   = compute_word_precision(pred, gt)
        r   = compute_word_recall(pred, gt)
        bl  = compute_bleu_scores(pred, gt)
        rl  = compute_rouge_l(pred, gt)
        met = compute_meteor(pred, gt)

        em_l.append(em); f1_l.append(f1); prec_l.append(p); rec_l.append(r)
        bl1_l.append(bl['bleu_1']); bl2_l.append(bl['bleu_2'])
        bl3_l.append(bl['bleu_3']); bl4_l.append(bl['bleu_4'])
        rl_l.append(rl); met_l.append(met)

        key = f"level_{comp}"
        if key not in per_comp:
            per_comp[key] = {'em':[], 'f1':[], 'rec':[], 'bl1':[], 'bl4':[], 'rl':[], 'met':[]}
        per_comp[key]['em'].append(em); per_comp[key]['f1'].append(f1)
        per_comp[key]['rec'].append(r); per_comp[key]['bl1'].append(bl['bleu_1'])
        per_comp[key]['bl4'].append(bl['bleu_4']); per_comp[key]['rl'].append(rl)
        per_comp[key]['met'].append(met)

    # BERTScore
    try:
        from bert_score import score as bert_score_fn
        _, _, bs_F1 = bert_score_fn(predictions, ground_truths, lang="en", verbose=False, rescale_with_baseline=True)
        bertscore = bs_F1.numpy().tolist()
    except Exception:
        bertscore = [0.0] * len(predictions)

    n = len(predictions)
    results = {
        'label': label, 'n': n,
        'em': em_l, 'f1': f1_l, 'prec': prec_l, 'rec': rec_l,
        'bl1': bl1_l, 'bl2': bl2_l, 'bl3': bl3_l, 'bl4': bl4_l,
        'rl': rl_l, 'met': met_l, 'bertscore': bertscore,
        'per_complexity': per_comp,
        'avg': {
            'exact_match': np.mean(em_l)*100, 'word_f1': np.mean(f1_l)*100,
            'word_precision': np.mean(prec_l)*100, 'word_recall': np.mean(rec_l)*100,
            'bleu_1': np.mean(bl1_l)*100, 'bleu_2': np.mean(bl2_l)*100,
            'bleu_3': np.mean(bl3_l)*100, 'bleu_4': np.mean(bl4_l)*100,
            'rouge_l': np.mean(rl_l)*100, 'meteor': np.mean(met_l)*100,
            'bertscore_f1': np.mean(bertscore)*100,
        },
    }
    return results

def print_results(results):
    a = results['avg']
    n = results['n']
    print(f"  Samples:          {n}")
    print(f"  Exact Match:      {a['exact_match']:.1f}%")
    print(f"  \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    print(f"  Word F1:          {a['word_f1']:.1f}%")
    print(f"  Word Precision:   {a['word_precision']:.1f}%")
    print(f"  Word Recall:      {a['word_recall']:.1f}%")
    print(f"  \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    print(f"  BLEU-1:           {a['bleu_1']:.1f}%")
    print(f"  BLEU-4:           {a['bleu_4']:.1f}%")
    print(f"  ROUGE-L:          {a['rouge_l']:.1f}%")
    print(f"  METEOR:           {a['meteor']:.1f}%")
    print(f"  BERTScore F1:     {a['bertscore_f1']:.1f}%")

    pc = results['per_complexity']
    print(f"\n  Per-Complexity:")
    for key in sorted(pc.keys()):
        s = pc[key]
        print(f"    {key}: F1={np.mean(s['f1'])*100:.1f}%, BLEU-1={np.mean(s['bl1'])*100:.1f}%, ROUGE-L={np.mean(s['rl'])*100:.1f}% [n={len(s['em'])}]")

# ── Inference helper ──
def run_inference(model, processor, eval_df, image_dir, max_tokens=64):
    """Run greedy inference on eval samples. Returns (predictions, ground_truths, complexities, questions, img_ids)."""
    preds, gts, comps, qs, ids = [], [], [], [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Inference"):
        img_path = Path(image_dir) / f"{row['img_id']}.jpg"
        if not img_path.exists(): continue

        image = Image.open(img_path).convert('RGB')
        question = str(row['question'])
        gt = str(row['answer'])

        prompt = f"Question: {question} Answer:"
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, dtype=torch.float16)

        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, num_beams=3)

        prompt_len = inputs['input_ids'].shape[1]
        new_tokens = generated[0][prompt_len:]
        prediction = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        if not prediction:
            prediction = processor.tokenizer.decode(generated[0], skip_special_tokens=True).strip()
            if "Answer:" in prediction:
                prediction = prediction.split("Answer:")[-1].strip()

        preds.append(prediction); gts.append(gt)
        comps.append(int(row.get('complexity', 1))); qs.append(question)
        ids.append(row['img_id'])
    return preds, gts, comps, qs, ids

print("All functions loaded. \u2713")

---
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 1: Baseline (Zero-Shot BLIP-2)
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

if SKIP_BASELINE:
    print("\u23e9 SKIPPING Phase 1 \u2014 Loading saved baseline results...")
    with open(f"{PRED_DIR}/baseline_summary.json") as f:
        baseline_saved = json.load(f)
    baseline_results = {'avg': baseline_saved['metrics'], 'label': 'Baseline', 'n': baseline_saved['num_eval_samples'], 'per_complexity': baseline_saved.get('per_complexity', {})}
    baseline_preds_df = pd.read_csv(f"{PRED_DIR}/baseline_predictions.csv")
    print(f"  Loaded: Word F1 = {baseline_saved['metrics']['word_f1']:.1f}%")
else:
    print("\ud83d\udd35 PHASE 1: Baseline Inference (Zero-Shot BLIP-2)")
    print("="*60)

    print(f"GPU: {torch.cuda.get_device_name(0)}")

    # Load base model (NO LoRA yet \u2014 pure zero-shot)
    processor = Blip2Processor.from_pretrained(MODEL_NAME)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    print(f"Loading {MODEL_NAME} in 8-bit...")
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map="auto", torch_dtype=torch.float16,
    )
    model.eval()
    print("Model loaded.")

    # Run baseline inference
    bl_preds, bl_gts, bl_comps, bl_qs, bl_ids = run_inference(
        model, processor, eval_subset, IMAGE_DIR, MAX_NEW_TOKENS
    )

    # Evaluate
    print("\nComputing baseline metrics...")
    baseline_results = evaluate_predictions(bl_preds, bl_gts, bl_comps, label="Baseline (Zero-Shot)")

    print(f"\n{'='*60}")
    print(f"  BASELINE RESULTS (Zero-Shot BLIP-2)")
    print(f"{'='*60}")
    print_results(baseline_results)
    print(f"{'='*60}")

    # Save
    baseline_summary = {
        'model': MODEL_NAME, 'method': 'zero-shot',
        'num_eval_samples': baseline_results['n'],
        'metrics': {k: round(v, 2) for k, v in baseline_results['avg'].items()},
        'per_complexity': {
            k: {mk: round(np.mean(mv)*100, 2) for mk, mv in v.items()}
            for k, v in baseline_results['per_complexity'].items()
        },
    }
    with open(f"{PRED_DIR}/baseline_summary.json", 'w') as f:
        json.dump(baseline_summary, f, indent=2)

    baseline_preds_df = pd.DataFrame({
        'img_id': bl_ids, 'question': bl_qs, 'ground_truth': bl_gts,
        'prediction': bl_preds, 'complexity': bl_comps,
        'word_f1': [round(x,3) for x in baseline_results['f1']],
        'bleu_1': [round(x,3) for x in baseline_results['bl1']],
        'rouge_l': [round(x,3) for x in baseline_results['rl']],
        'meteor': [round(x,3) for x in baseline_results['met']],
        'bertscore_f1': [round(x,3) for x in baseline_results['bertscore']],
    })
    baseline_preds_df.to_csv(f"{PRED_DIR}/baseline_predictions.csv", index=False)
    print(f"\n\u2713 Saved to {PRED_DIR}/baseline_*.")

---
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 2: LoRA Fine-Tuning
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550

### Cell 7: Prepare LoRA Model

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, TaskType

if SKIP_TRAINING:
    print("\u23e9 SKIPPING Phase 2 \u2014 Loading LoRA checkpoint...")

    # If model not loaded yet (e.g., baseline was also skipped)
    if 'model' not in dir() or model is None:
        processor = Blip2Processor.from_pretrained(MODEL_NAME)
        if processor.tokenizer.pad_token is None:
            processor.tokenizer.pad_token = processor.tokenizer.eos_token
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = Blip2ForConditionalGeneration.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.float16,
        )

    # Find and load LoRA checkpoint
    lora_path = None
    for name in ["best_lora", "final_lora"]:
        p = f"{CKPT_DIR}/{name}"
        if os.path.exists(p):
            lora_path = p
            break

    if lora_path:
        model = PeftModel.from_pretrained(model, lora_path)
        print(f"  LoRA loaded from {lora_path}")
    else:
        print("  \u26a0\ufe0f No LoRA checkpoint found! Using base model.")

    model.eval()

    # Load saved fine-tuned results if available
    ft_summary_path = f"{PRED_DIR}/finetuned_summary.json"
    if os.path.exists(ft_summary_path):
        with open(ft_summary_path) as f:
            ft_saved = json.load(f)
        finetuned_results = {'avg': ft_saved.get('metrics', {}), 'label': 'Fine-Tuned', 'n': ft_saved.get('num_eval_samples', 0), 'per_complexity': ft_saved.get('per_complexity', {})}
        print(f"  Loaded saved results: Word F1 = {finetuned_results['avg'].get('word_f1', 'N/A')}%")
    else:
        finetuned_results = None
        print("  No saved fine-tuned results \u2014 will re-evaluate in next cell.")

else:
    print("\ud83d\udfe2 PHASE 2: LoRA Fine-Tuning")
    print("="*60)

    # If model not loaded yet
    if 'processor' not in dir():
        processor = Blip2Processor.from_pretrained(MODEL_NAME)
        if processor.tokenizer.pad_token is None:
            processor.tokenizer.pad_token = processor.tokenizer.eos_token
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = Blip2ForConditionalGeneration.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.float16,
        )

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")
    finetuned_results = None  # Will be computed after training

### Cell 8: Training Loop

In [ ]:
if not SKIP_TRAINING:
    from torch.utils.data import Dataset, DataLoader
    from transformers import get_cosine_schedule_with_warmup

    # Stratified training subset
    np.random.seed(SEED)
    if 'complexity' in train_df.columns:
        levels = sorted(train_df['complexity'].unique())
        per_level = TRAIN_SUBSET_SIZE // len(levels)
        subsets = [train_df[train_df['complexity'] == lvl].sample(
            n=min(per_level, len(train_df[train_df['complexity'] == lvl])), random_state=SEED
        ) for lvl in levels]
        train_subset = pd.concat(subsets).sample(frac=1, random_state=SEED).reset_index(drop=True)
    else:
        train_subset = train_df.sample(n=min(TRAIN_SUBSET_SIZE, len(train_df)), random_state=SEED)

    print(f"Training subset: {len(train_subset):,} samples")

    # Dataset class
    class VQAFineTuneDataset(Dataset):
        def __init__(self, df, image_dir, processor, max_length=128):
            self.df = df.reset_index(drop=True)
            self.image_dir = Path(image_dir)
            self.processor = processor
            self.max_length = max_length
            valid = self.df['img_id'].apply(lambda x: (self.image_dir / f"{x}.jpg").exists())
            self.df = self.df[valid].reset_index(drop=True)
            print(f"  Dataset: {len(self.df)} samples (after image check)")

        def __len__(self): return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            image = Image.open(self.image_dir / f"{row['img_id']}.jpg").convert('RGB')
            question, answer = str(row['question']), str(row['answer'])

            prompt_text = f"Question: {question} Answer:"
            full_text   = f"Question: {question} Answer: {answer}"

            prompt_len = self.processor.tokenizer(
                prompt_text, add_special_tokens=False, return_tensors="pt"
            ).input_ids.shape[1]

            encoding = self.processor(
                images=image, text=full_text, return_tensors="pt",
                padding="max_length", truncation=True, max_length=self.max_length,
            )

            input_ids      = encoding.input_ids.squeeze()
            attention_mask = encoding.attention_mask.squeeze()
            pixel_values   = encoding.pixel_values.squeeze()

            labels = input_ids.clone()
            labels[:prompt_len] = -100
            labels[labels == self.processor.tokenizer.pad_token_id] = -100

            return {'pixel_values': pixel_values, 'input_ids': input_ids,
                    'attention_mask': attention_mask, 'labels': labels}

    train_dataset = VQAFineTuneDataset(train_subset, IMAGE_DIR, processor)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    total_steps  = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    print(f"Effective batch: {BATCH_SIZE}x{GRAD_ACCUM_STEPS}={BATCH_SIZE*GRAD_ACCUM_STEPS} | Steps: {total_steps} | Warmup: {warmup_steps}")

    # \u2500\u2500 TRAINING LOOP \u2500\u2500
    model.train()
    training_log = []
    best_loss = float('inf')
    start_time = time.time()

    for epoch in range(EPOCHS):
        epoch_loss, step_count = 0.0, 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for step, batch in enumerate(pbar):
            pixel_values   = batch['pixel_values'].to(model.device, dtype=torch.float16)
            input_ids      = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            labels         = batch['labels'].to(model.device)

            outputs = model(pixel_values=pixel_values, input_ids=input_ids,
                            attention_mask=attention_mask, labels=labels)

            loss = outputs.loss / GRAD_ACCUM_STEPS
            loss.backward()
            epoch_loss += outputs.loss.item()
            step_count += 1

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            pbar.set_postfix({'loss': f'{epoch_loss/step_count:.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

        avg_loss = epoch_loss / step_count
        elapsed  = time.time() - start_time
        training_log.append({'epoch': epoch+1, 'avg_loss': avg_loss, 'elapsed_min': elapsed/60})
        print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, Time={elapsed/60:.1f}min")

        if avg_loss < best_loss:
            best_loss = avg_loss
            model.save_pretrained(f"{CKPT_DIR}/best_lora")
            print(f"  \u2713 Best checkpoint (loss={best_loss:.4f})")

    model.save_pretrained(f"{CKPT_DIR}/final_lora")
    with open(f"{RESULTS_DIR}/training_log.json", 'w') as f:
        json.dump(training_log, f, indent=2)

    print(f"\n\u2713 Training done! {(time.time()-start_time)/60:.1f}min | Best loss: {best_loss:.4f}")
    print(f"  Checkpoints at {CKPT_DIR}")

else:
    print("\u23e9 Training skipped.")

### Cell 9: Evaluate Fine-Tuned Model

In [ ]:
if finetuned_results is None:
    print("Evaluating fine-tuned model...")
    model.eval()

    ft_preds, ft_gts, ft_comps, ft_qs, ft_ids = run_inference(
        model, processor, eval_subset, IMAGE_DIR, MAX_NEW_TOKENS
    )

    finetuned_results = evaluate_predictions(ft_preds, ft_gts, ft_comps, label="Fine-Tuned (LoRA)")

    print(f"\n{'='*60}")
    print(f"  FINE-TUNED RESULTS (LoRA BLIP-2)")
    print(f"{'='*60}")
    print_results(finetuned_results)
    print(f"{'='*60}")

    # Save
    ft_summary = {
        'model': MODEL_NAME, 'method': 'LoRA fine-tuned',
        'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA,
        'training_samples': TRAIN_SUBSET_SIZE,
        'epochs': EPOCHS, 'num_eval_samples': finetuned_results['n'],
        'metrics': {k: round(v, 2) for k, v in finetuned_results['avg'].items()},
        'per_complexity': {
            k: {mk: round(np.mean(mv)*100, 2) for mk, mv in v.items()}
            for k, v in finetuned_results['per_complexity'].items()
        },
    }
    with open(f"{PRED_DIR}/finetuned_summary.json", 'w') as f:
        json.dump(ft_summary, f, indent=2)

    ft_preds_df = pd.DataFrame({
        'img_id': ft_ids, 'question': ft_qs, 'ground_truth': ft_gts,
        'prediction': ft_preds, 'complexity': ft_comps,
        'word_f1': [round(x,3) for x in finetuned_results['f1']],
        'bleu_1': [round(x,3) for x in finetuned_results['bl1']],
        'rouge_l': [round(x,3) for x in finetuned_results['rl']],
        'meteor': [round(x,3) for x in finetuned_results['met']],
    })
    ft_preds_df.to_csv(f"{PRED_DIR}/finetuned_predictions.csv", index=False)
    print(f"\u2713 Saved to {PRED_DIR}/finetuned_*.")
else:
    print(f"Fine-tuned results loaded. Word F1 = {finetuned_results['avg'].get('word_f1', 'N/A')}%")

---
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 3: Uncertainty Estimation & Abstention
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550

### Cell 10: Uncertainty Functions

In [ ]:
# \u2500\u2500 Predictive Entropy + Sequence Log-prob (single pass) \u2500\u2500
def get_entropy_and_confidence(model, processor, image, question, max_tokens=64):
    """Returns (prediction, entropy_mean, confidence)."""
    prompt = f"Question: {question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, dtype=torch.float16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, do_sample=False,
            output_scores=True, return_dict_in_generate=True,
        )

    prompt_len = inputs['input_ids'].shape[1]
    gen_ids = outputs.sequences[0][prompt_len:]
    prediction = processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    entropies, log_probs = [], []
    for step, score in enumerate(outputs.scores):
        probs = torch.softmax(score[0], dim=-1)
        log_p = torch.log(probs.clamp(min=1e-10))
        entropies.append(-(probs * log_p).sum().item())
        if step < len(gen_ids):
            log_probs.append(log_p[gen_ids[step]].item())

    entropy_mean = float(np.mean(entropies)) if entropies else 0.0
    confidence = float(np.exp(np.mean(log_probs))) if log_probs else 0.0
    return prediction, entropy_mean, confidence


# \u2500\u2500 MC Dropout (N stochastic passes) \u2500\u2500
def enable_dropout(m):
    for mod in m.modules():
        if isinstance(mod, torch.nn.Dropout): mod.train()

def disable_dropout(m):
    for mod in m.modules():
        if isinstance(mod, torch.nn.Dropout): mod.eval()

def mc_dropout_inference(model, processor, image, question, n_passes=5, max_tokens=64):
    """Returns (majority_answer, all_answers, mc_uncertainty, unique_ratio)."""
    prompt = f"Question: {question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, dtype=torch.float16)
    prompt_len = inputs['input_ids'].shape[1]

    enable_dropout(model)
    answers = []
    for _ in range(n_passes):
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        ans = processor.tokenizer.decode(gen[0][prompt_len:], skip_special_tokens=True).strip()
        answers.append(ans)
    disable_dropout(model)

    # Pairwise F1 -> variance
    pw = [compute_word_f1(answers[i], answers[j]) for i in range(len(answers)) for j in range(i+1, len(answers))]
    mc_unc = 1.0 - (np.mean(pw) if pw else 1.0)

    normed = [normalize_text(a) for a in answers]
    majority = Counter(normed).most_common(1)[0][0]
    prediction = answers[normed.index(majority)]

    return prediction, answers, float(mc_unc), len(set(normed))/len(normed)

print("Uncertainty functions loaded. \u2713")

### Cell 11: Run Uncertainty-Aware Inference

In [ ]:
if SKIP_UNCERTAINTY:
    print("\u23e9 SKIPPING Phase 3 \u2014 Loading saved uncertainty results...")
    unc_csv_path = f"{UNC_DIR}/uncertainty_predictions.csv"
    if os.path.exists(unc_csv_path):
        unc_df = pd.read_csv(unc_csv_path)
        unc_results = [unc_df.iloc[i].to_dict() for i in range(len(unc_df))]
        print(f"  Loaded {len(unc_results)} results.")
    else:
        print("  \u26a0\ufe0f No saved uncertainty results found!")
        unc_results = []
else:
    print("\ud83d\udd34 PHASE 3: Uncertainty Estimation")
    print("="*60)
    print(f"  Entropy + LogProb: 1 pass/sample")
    print(f"  MC Dropout: {MC_DROPOUT_PASSES} passes/sample")
    print(f"  Total: ~{len(eval_subset) * (1 + MC_DROPOUT_PASSES)} generations\n")

    model.eval()
    unc_results = []
    start = time.time()

    for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc="Uncertainty eval"):
        img_path = Path(IMAGE_DIR) / f"{row['img_id']}.jpg"
        if not img_path.exists(): continue

        image = Image.open(img_path).convert('RGB')
        question = str(row['question'])
        gt = str(row['answer'])
        comp = int(row.get('complexity', 1))

        # 1) Entropy + confidence
        pred_greedy, entropy, confidence = get_entropy_and_confidence(
            model, processor, image, question, MAX_NEW_TOKENS
        )

        # 2) MC Dropout
        pred_mc, mc_answers, mc_unc, unique_ratio = mc_dropout_inference(
            model, processor, image, question, MC_DROPOUT_PASSES, MAX_NEW_TOKENS
        )

        prediction = pred_mc  # majority answer

        # Combined uncertainty
        entropy_norm = min(entropy / 10.0, 1.0)
        conf_unc = 1.0 - confidence
        combined = 0.4 * entropy_norm + 0.3 * mc_unc + 0.3 * conf_unc

        # Metrics
        f1 = compute_word_f1(prediction, gt)
        em = int(compute_exact_match(prediction, gt))
        bl = compute_bleu_scores(prediction, gt)
        rl = compute_rouge_l(prediction, gt)
        met = compute_meteor(prediction, gt)

        unc_results.append({
            'img_id': row['img_id'], 'question': question, 'ground_truth': gt,
            'prediction': prediction, 'complexity': comp,
            'exact_match': em, 'word_f1': f1,
            'bleu_1': bl['bleu_1'], 'bleu_4': bl['bleu_4'],
            'rouge_l': rl, 'meteor': met,
            'entropy': entropy, 'confidence': confidence,
            'mc_uncertainty': mc_unc, 'mc_unique_ratio': unique_ratio,
            'combined_uncertainty': combined,
        })

    elapsed = time.time() - start
    print(f"\n\u2713 Done! {elapsed/60:.1f}min for {len(unc_results)} samples ({elapsed/max(1,len(unc_results)):.1f}s/sample)")

### Cell 12: Abstention & Safety Analysis

In [ ]:
if len(unc_results) == 0:
    print("\u26a0\ufe0f No uncertainty results available. Skipping safety analysis.")
else:
    f1_scores     = [r['word_f1'] for r in unc_results]
    entropies     = [r['entropy'] for r in unc_results]
    confidences   = [r['confidence'] for r in unc_results]
    mc_uncs       = [r['mc_uncertainty'] for r in unc_results]
    combined_uncs = [r['combined_uncertainty'] for r in unc_results]
    comps         = [r['complexity'] for r in unc_results]

    # \u2500\u2500 Correlation \u2500\u2500
    print("Correlation (uncertainty vs F1, negative = good):")
    for name, vals in [('Entropy', entropies), ('MC Dropout', mc_uncs),
                       ('1-Confidence', [1-c for c in confidences]), ('Combined', combined_uncs)]:
        r = np.corrcoef(vals, f1_scores)[0,1]
        print(f"  {name:<15} r = {r:+.3f}")

    # \u2500\u2500 Threshold tuning \u2500\u2500
    unc_arr = np.array(combined_uncs)
    f1_arr  = np.array(f1_scores)

    thresholds = np.linspace(unc_arr.min(), unc_arr.max(), 100)
    best_t, best_acc, best_cov = thresholds[-1], 0, 1.0
    for t in thresholds:
        mask = unc_arr <= t
        cov  = mask.sum() / len(unc_arr)
        sel  = f1_arr[mask].mean() if mask.sum() > 0 else 0
        if cov >= TARGET_COVERAGE and sel > best_acc:
            best_t, best_acc, best_cov = t, sel, cov

    answered  = [i for i, u in enumerate(combined_uncs) if u <= best_t]
    abstained = [i for i, u in enumerate(combined_uncs) if u > best_t]

    print(f"\n{'='*60}")
    print(f"  ABSTENTION RESULTS")
    print(f"{'='*60}")
    print(f"  Threshold \u03c4:      {best_t:.4f}")
    print(f"  Coverage:         {best_cov*100:.1f}% ({len(answered)}/{len(unc_results)})")
    print(f"  Selective F1:     {best_acc*100:.1f}%")
    print(f"  Overall F1:       {np.mean(f1_scores)*100:.1f}%")
    print(f"  Improvement:      +{(best_acc - np.mean(f1_scores))*100:.1f}%")
    print(f"  Abstained:        {len(abstained)} samples")

    # Per-complexity abstention
    print(f"\n  Abstention by Complexity:")
    for lvl in sorted(set(comps)):
        idx = [i for i, c in enumerate(comps) if c == lvl]
        abs_count = sum(1 for i in idx if combined_uncs[i] > best_t)
        print(f"    Level {lvl}: {abs_count}/{len(idx)} abstained ({abs_count/len(idx)*100:.0f}%)")

    # \u2500\u2500 AUROC \u2500\u2500
    binary_correct = (f1_arr >= 0.5).astype(int)
    inc_idx = np.where(binary_correct == 0)[0]
    cor_idx = np.where(binary_correct == 1)[0]
    if len(inc_idx) > 0 and len(cor_idx) > 0:
        concordant = sum(
            1 if unc_arr[i] > unc_arr[j] else 0.5 if unc_arr[i] == unc_arr[j] else 0
            for i in inc_idx for j in cor_idx
        )
        auroc = concordant / (len(inc_idx) * len(cor_idx))
    else:
        auroc = 0.5

    # \u2500\u2500 Risk-Coverage \u2500\u2500
    sorted_idx = np.argsort(unc_arr)
    sorted_f1  = f1_arr[sorted_idx]
    coverages  = [(n+1)/len(sorted_f1) for n in range(len(sorted_f1))]
    sel_accs   = [sorted_f1[:n+1].mean() for n in range(len(sorted_f1))]
    risks      = [1 - a for a in sel_accs]
    auc_risk   = float(np.trapz(risks, coverages))

    # \u2500\u2500 ECE \u2500\u2500
    conf_arr = np.array(confidences)
    n_bins = 10
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0; ece_data = []
    for i in range(n_bins):
        mask = (conf_arr >= bins[i]) & (conf_arr < bins[i+1]) if i < n_bins-1 else (conf_arr >= bins[i]) & (conf_arr <= bins[i+1])
        nb = mask.sum()
        if nb == 0:
            ece_data.append((0,0,0)); continue
        ac, ag = conf_arr[mask].mean(), f1_arr[mask].mean()
        ece += (nb/len(conf_arr)) * abs(ag - ac)
        ece_data.append((float(ac), float(ag), int(nb)))

    # \u2500\u2500 Selective accuracy table \u2500\u2500
    sel_acc_table = {}
    for target in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        n = max(1, int(target * len(sorted_f1)))
        sel_acc_table[target] = float(sorted_f1[:n].mean())

    print(f"\n{'='*60}")
    print(f"  SAFETY METRICS")
    print(f"{'='*60}")
    print(f"  AUROC:       {auroc:.3f}  {'(good)' if auroc > 0.6 else '(fair)' if auroc > 0.5 else '(poor)'}")
    print(f"  AUC-Risk:    {auc_risk:.3f}  (lower is better)")
    print(f"  ECE:         {ece:.3f}  (lower is better)")
    print(f"\n  Selective Accuracy:")
    for cov_k, acc_v in sel_acc_table.items():
        m = " \u2190 target" if abs(cov_k - TARGET_COVERAGE) < 0.01 else ""
        print(f"    {cov_k*100:>5.0f}% coverage \u2192 {acc_v*100:.1f}% F1{m}")
    print(f"{'='*60}")

### Cell 13: Safety Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11

if len(unc_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))

    # 1) Selective Accuracy vs Coverage
    ax = axes[0, 0]
    ax.plot(coverages, [a*100 for a in sel_accs], 'b-', lw=2, label='Selective F1')
    ax.axhline(y=np.mean(f1_scores)*100, color='r', ls='--', alpha=.7, label=f'Overall F1 ({np.mean(f1_scores)*100:.1f}%)')
    ax.axvline(x=TARGET_COVERAGE, color='g', ls=':', alpha=.7, label=f'Target ({TARGET_COVERAGE*100:.0f}%)')
    ax.set_xlabel('Coverage'); ax.set_ylabel('Selective Word F1 (%)')
    ax.set_title(f'Risk-Coverage (AUROC={auroc:.3f})'); ax.legend(fontsize=9); ax.grid(True, alpha=.3)

    # 2) Uncertainty vs F1 scatter
    ax = axes[0, 1]
    colors = ['#2ecc71' if f >= 0.5 else '#e74c3c' for f in f1_scores]
    ax.scatter(combined_uncs, [f*100 for f in f1_scores], c=colors, alpha=.7, s=50, edgecolors='w', lw=.5)
    ax.axvline(x=best_t, color='orange', ls='--', lw=2, label=f'\u03c4={best_t:.3f}')
    ax.set_xlabel('Combined Uncertainty'); ax.set_ylabel('Word F1 (%)')
    ax.set_title('Uncertainty vs Quality'); ax.legend(fontsize=9); ax.grid(True, alpha=.3)

    # 3) Reliability diagram
    ax = axes[1, 0]
    bc = [d[0] for d in ece_data if d[2] > 0]
    ba = [d[1] for d in ece_data if d[2] > 0]
    ax.bar(bc, ba, width=.08, alpha=.7, color='#3498db', label='Actual accuracy')
    ax.plot([0,1], [0,1], 'r--', alpha=.7, label='Perfect calibration')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy (F1)')
    ax.set_title(f'Reliability Diagram (ECE={ece:.3f})'); ax.legend(fontsize=9)
    ax.grid(True, alpha=.3); ax.set_xlim(-.05,1.05); ax.set_ylim(-.05,1.05)

    # 4) Uncertainty distribution by complexity
    ax = axes[1, 1]
    for lvl in sorted(set(comps)):
        lvl_u = [combined_uncs[i] for i, c in enumerate(comps) if c == lvl]
        ax.hist(lvl_u, bins=15, alpha=.5, label=f'Level {lvl} (n={len(lvl_u)})')
    ax.axvline(x=best_t, color='orange', ls='--', lw=2, label=f'\u03c4={best_t:.3f}')
    ax.set_xlabel('Combined Uncertainty'); ax.set_ylabel('Count')
    ax.set_title('Uncertainty by Complexity'); ax.legend(fontsize=9); ax.grid(True, alpha=.3)

    plt.suptitle('Uncertainty-Aware Medical VQA \u2014 Safety Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{UNC_DIR}/safety_plots.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved \u2192 {UNC_DIR}/safety_plots.png")
else:
    print("No uncertainty data \u2014 skipping plots.")

---
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 4: Final Comparison & Results
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550

### Cell 14: Side-by-Side Comparison Table

In [ ]:
print("\n" + "="*80)
print("  FINAL COMPARISON: Baseline vs Fine-Tuned vs Uncertainty-Aware")
print("="*80)

# Collect metrics
bl_avg = baseline_results.get('avg', {})
ft_avg = finetuned_results.get('avg', {}) if finetuned_results else {}

if len(unc_results) > 0:
    unc_avg = {
        'word_f1': np.mean(f1_scores)*100,
        'exact_match': np.mean([r['exact_match'] for r in unc_results])*100,
        'bleu_1': np.mean([r['bleu_1'] for r in unc_results])*100,
        'bleu_4': np.mean([r['bleu_4'] for r in unc_results])*100,
        'rouge_l': np.mean([r['rouge_l'] for r in unc_results])*100,
        'meteor': np.mean([r['meteor'] for r in unc_results])*100,
    }
    sel_f1 = best_acc * 100
else:
    unc_avg = {}; sel_f1 = 0

# Print table
def val(d, k, default='-'): return f"{d[k]:.1f}%" if k in d and d[k] is not None else default

metrics = [
    ('Exact Match', 'exact_match'),
    ('Word F1', 'word_f1'),
    ('BLEU-1', 'bleu_1'),
    ('BLEU-4', 'bleu_4'),
    ('ROUGE-L', 'rouge_l'),
    ('METEOR', 'meteor'),
    ('BERTScore F1', 'bertscore_f1'),
]

print(f"  {'Metric':<18} {'Baseline':>12} {'Fine-Tuned':>12} {'Unc-Aware':>12} {'Selective':>12}")
print(f"  {'-'*68}")
for name, key in metrics:
    b = val(bl_avg, key)
    f = val(ft_avg, key)
    u = val(unc_avg, key)
    # Selective = uncertainty-aware at 80% coverage
    if key == 'word_f1' and len(unc_results) > 0:
        s = f"{sel_f1:.1f}%"
    else:
        s = '-'
    print(f"  {name:<18} {b:>12} {f:>12} {u:>12} {s:>12}")

if len(unc_results) > 0:
    print(f"\n  {'Safety Metrics':>18}")
    print(f"  {'-'*40}")
    print(f"  {'Coverage':<18} {'-':>12} {'-':>12} {best_cov*100:>11.1f}%")
    print(f"  {'AUROC':<18} {'-':>12} {'-':>12} {auroc:>12.3f}")
    print(f"  {'AUC-Risk':<18} {'-':>12} {'-':>12} {auc_risk:>12.3f}")
    print(f"  {'ECE':<18} {'-':>12} {'-':>12} {ece:>12.3f}")

print("="*80)

if len(unc_results) > 0:
    print(f"\n  Key Takeaway:")
    print(f"  \u2022 Selective F1 ({sel_f1:.1f}%) > Overall F1 ({np.mean(f1_scores)*100:.1f}%)")
    print(f"  \u2022 Model abstains on {len(abstained)} hardest samples \u2192 higher accuracy on the rest")
    print(f"  \u2022 AUROC={auroc:.3f} \u2192 uncertainty {'is' if auroc > 0.5 else 'is NOT'} informative")

### Cell 15: Sample Predictions with Uncertainty

In [ ]:
print("\nSample Predictions")
print("="*90)

data = unc_results if len(unc_results) > 0 else []
for i in range(min(15, len(data))):
    r = data[i]
    f1 = r['word_f1']
    unc = r.get('combined_uncertainty', 0)
    will_abstain = unc > best_t if len(unc_results) > 0 else False

    if will_abstain:
        label = "\ud83d\udeab ABSTAIN"
    elif r.get('exact_match', 0):
        label = "\u2713 EXACT"
    elif f1 >= 0.5:
        label = "~ PARTIAL"
    else:
        label = "\u2717 WRONG"

    print(f"[{i+1}] {label} | C{r['complexity']} | F1={f1:.2f} | Unc={unc:.3f} | Conf={r.get('confidence',0):.3f}")
    print(f"  Q:    {r['question'][:85]}")
    print(f"  GT:   {r['ground_truth'][:85]}")
    print(f"  Pred: {r['prediction'][:85]}")
    print("-"*90)

### Cell 16: Save All Results & Download

In [ ]:
# \u2500\u2500 Save uncertainty results \u2500\u2500
if len(unc_results) > 0:
    unc_summary = {
        'model': MODEL_NAME, 'method': 'LoRA + uncertainty',
        'mc_dropout_passes': MC_DROPOUT_PASSES,
        'target_coverage': TARGET_COVERAGE,
        'eval_samples': len(unc_results),
        'vqa_metrics': {k: round(v, 2) for k, v in unc_avg.items()},
        'safety_metrics': {
            'auroc': round(auroc, 4), 'auc_risk': round(auc_risk, 4), 'ece': round(ece, 4),
        },
        'abstention': {
            'threshold': round(best_t, 4), 'coverage': round(best_cov, 4),
            'selective_f1': round(sel_f1, 2), 'overall_f1': round(np.mean(f1_scores)*100, 2),
            'n_answered': len(answered), 'n_abstained': len(abstained),
        },
        'selective_accuracy': {f"{int(k*100)}pct": round(v*100, 2) for k, v in sel_acc_table.items()},
    }
    with open(f"{UNC_DIR}/uncertainty_summary.json", 'w') as f:
        json.dump(unc_summary, f, indent=2)

    unc_df = pd.DataFrame(unc_results)
    unc_df['abstained'] = [combined_uncs[i] > best_t for i in range(len(unc_results))]
    unc_df.to_csv(f"{UNC_DIR}/uncertainty_predictions.csv", index=False)

# \u2500\u2500 List all output files \u2500\u2500
print("\nAll output files:")
for folder in [PRED_DIR, UNC_DIR, CKPT_DIR]:
    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            for f in files:
                fp = os.path.join(root, f)
                print(f"  {fp} ({os.path.getsize(fp)/1024:.1f} KB)")

# \u2500\u2500 Download \u2500\u2500
if not USE_DRIVE:
    try:
        from google.colab import files
        for f in ['baseline_summary.json', 'finetuned_summary.json']:
            fp = f"{PRED_DIR}/{f}"
            if os.path.exists(fp): files.download(fp)
        if os.path.exists(f"{UNC_DIR}/uncertainty_summary.json"):
            files.download(f"{UNC_DIR}/uncertainty_summary.json")
        if os.path.exists(f"{UNC_DIR}/safety_plots.png"):
            files.download(f"{UNC_DIR}/safety_plots.png")
    except Exception as e:
        print(f"Download: {e}")
else:
    print("\nAll results saved to Google Drive.")

print("\n\ud83c\udf89 Pipeline complete!")